In [1]:
import os
from pathlib import Path
REPO = Path.cwd().resolve().parents[1]          # docs/notebooks -> repo root
GOLDEN = Path(os.environ.get("CAL_DISP_GOLDEN_DIR", REPO / "test_golden"))
SCRATCH = Path(os.environ.get("TMPDIR", REPO / "tmp")) / "cal_disp_notebooks"; SCRATCH.mkdir(parents=True, exist_ok=True)

# Fix 4 — geometric consistency check between the inputs

**Defect.** `src/cal_disp/workflow.py` `_load_los_bands` (gamma-release lines 105–118) read
the three LOS bands raw and the workflow then combined them with the DISP displacement by
array index. Nothing compared the shape, transform or CRS of the LOS/DEM rasters with the
DISP grid, nor `StaticLayer.frame_id` with `DispProduct.frame_id`: static layers of another
frame with the same shape would have projected the GNSS field onto the wrong line of sight
without any message. The LOS nodata value (0.0) was not masked on this path either.

**Fix.** New module `src/cal_disp/prep/consistency.py`:

* `check_input_geometry(disp, los_file, dem_file, tropo_files)` — called once at the start
  of `run_calibration` — requires equal shape, transform (each term within 1e-6 pixel),
  CRS and frame id, and raises a `ValueError` naming the offending file and values;
* `check_tropo_coverage` (called by `prepare_troposphere_correction`) requires each TROPO
  file to cover the frame;
* `mask_los_nodata` sets LOS pixels without data (all-zero vectors or the raster nodata
  value) to NaN; `_load_los_bands` applies it.

**Demonstrated here.** The check on the golden inputs (passes; the compared values are
printed), then on scratch copies of the LOS raster with a shifted transform, another CRS and
the name of another frame (each raises). The pre-fix loader, checked out from
`gamma-release` (commit `d0901b3`), accepts all three.

In [2]:
import importlib.util
import logging
import shutil
import subprocess

import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.transform import Affine

from cal_disp.prep.consistency import check_input_geometry
from cal_disp.product import DispProduct
from cal_disp.workflow import _load_los_bands, run_calibration

inp = GOLDEN / "input_data"
disp_file = next((inp / "disp").glob("OPERA_L3_DISP-S1_*.nc"))
los_file = next((inp / "static_input").glob("*line_of_sight_enu.tif"))
dem_file = next((inp / "static_input").glob("*_dem.tif"))
tropo_files = sorted((inp / "tropo").glob("OPERA_L4_TROPO-ZENITH_*.nc"))
lookup_file = next((inp / "gnss").glob("grid_latlon_lookup_v*.txt"))

work = SCRATCH / "11_input_consistency"
shutil.rmtree(work, ignore_errors=True)
work.mkdir()

/u/aurora-r0/govorcin/01_OPERA/VLM/DISP_CAL/gamma_release/cal-disp/.pixi/envs/dev/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Golden inputs: consistent

In [3]:
geometries = check_input_geometry(disp_file, los_file, dem_file, tropo_files=tropo_files)
for geom in geometries.values():
    print(geom.describe())
    print("   ", geom.path.name)
print("\nTROPO files cover the frame:", [p.name[:38] for p in tropo_files])
print("frame bounds (WGS84):", {k: round(v, 3) for k, v in DispProduct.from_path(disp_file).get_bounds_wgs84().items()})

DISP: frame F08882, shape (7733, 9464), EPSG:32615, origin (71970.000, 3385920.000), pixel (30, -30)
    OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20251027T005420Z.nc
LOS: frame F08882, shape (7733, 9464), EPSG:32615, origin (71970.000, 3385920.000), pixel (30, -30)
    OPERA_L3_DISP-S1-STATIC_F08882_20140403_S1A_v1.0_line_of_sight_enu.tif
DEM: frame F08882, shape (7733, 9464), EPSG:32615, origin (71970.000, 3385920.000), pixel (30, -30)
    OPERA_L3_DISP-S1-STATIC_F08882_20140403_S1A_v1.0_dem.tif

TROPO files cover the frame: ['OPERA_L4_TROPO-ZENITH_20220111T000000Z', 'OPERA_L4_TROPO-ZENITH_20220722T000000Z']
frame bounds (WGS84): {'west': -97.459, 'south': 28.442, 'east': -94.473, 'north': 30.597}


## Inconsistent inputs

Scratch copies of the golden LOS raster (20 MB): the georeferencing is rewritten in place
with `rasterio` (`r+`), the pixel values are untouched — so the array still has the right
shape and would have been used as is.

In [4]:
def los_copy(subdir: str, name: str = los_file.name) -> Path:
    path = work / subdir / name
    path.parent.mkdir(parents=True)
    shutil.copyfile(los_file, path)
    return path


# 1. Transform shifted by 10 pixels east and 0.5 pixel south
shifted = los_copy("shifted_transform")
with rasterio.open(shifted, "r+") as dst:
    dst.transform = Affine.translation(300.0, -15.0) * dst.transform

# 2. Another CRS (UTM zone 16N instead of 15N), same numbers
other_crs = los_copy("other_crs")
with rasterio.open(other_crs, "r+") as dst:
    dst.crs = CRS.from_epsg(32616)

# 3. The static layer of another frame: same grid, another frame id in the name
other_frame = los_copy("other_frame", los_file.name.replace("F08882", "F08622"))

cases = {
    "shifted transform": shifted,
    "different CRS": other_crs,
    "other frame id": other_frame,
}
for label, path in cases.items():
    with rasterio.open(path) as src:
        print(f"{label:18s} shape={src.shape} crs=EPSG:{src.crs.to_epsg()} origin=({src.transform.c}, {src.transform.f})  {path.name[24:30]}")

shifted transform  shape=(7733, 9464) crs=EPSG:32615 origin=(72270.0, 3385905.0)  F08882
different CRS      shape=(7733, 9464) crs=EPSG:32616 origin=(71970.0, 3385920.0)  F08882
other frame id     shape=(7733, 9464) crs=EPSG:32615 origin=(71970.0, 3385920.0)  F08622


In [5]:
for label, path in cases.items():
    try:
        check_input_geometry(disp_file, path, dem_file)
        print(f"{label}: passed (unexpected)")
    except ValueError as exc:
        print(f"{label}: ValueError")
        print("   ", str(exc).replace(str(SCRATCH), "<scratch>"), "\n")

shifted transform: ValueError
    LOS file <scratch>/11_input_consistency/shifted_transform/OPERA_L3_DISP-S1-STATIC_F08882_20140403_S1A_v1.0_line_of_sight_enu.tif is not on the grid of DISP file OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20251027T005420Z.nc: transform (origin (72270.0, 3385905.0), pixel (30.0, -30.0)) != (origin (71970.0, 3385920.0), pixel (30.0, -30.0)); origin shifted by (10, 0.5) pixels 

different CRS: ValueError
    LOS file <scratch>/11_input_consistency/other_crs/OPERA_L3_DISP-S1-STATIC_F08882_20140403_S1A_v1.0_line_of_sight_enu.tif is not on the grid of DISP file OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20251027T005420Z.nc: CRS EPSG:32616 != EPSG:32615 



other frame id: ValueError
    LOS file <scratch>/11_input_consistency/other_frame/OPERA_L3_DISP-S1-STATIC_F08622_20140403_S1A_v1.0_line_of_sight_enu.tif is not on the grid of DISP file OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20251027T005420Z.nc: frame id 8622 != 8882 



## Before / after in the workflow

The pre-fix `workflow.py` is taken from `gamma-release` before this fix (commit `d0901b3`). Its LOS loader — the only place the
LOS file was opened before the fit — returns the arrays of all three inconsistent files
without complaint, and its source contains no geometry comparison. The fixed
`run_calibration` stops at once, before any GNSS or fitting work.

In [6]:
old_src = subprocess.run(
    ["git", "show", "d0901b3:src/cal_disp/workflow.py"],
    cwd=REPO, check=True, capture_output=True, text=True,
).stdout
old_path = work / "workflow_old.py"
old_path.write_text(old_src)
spec = importlib.util.spec_from_file_location("workflow_old", old_path)
workflow_old = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow_old)

print("geometry check in the old workflow source:", "check_input_geometry" in old_src or "frame_id !=" in old_src)
print()
logging.getLogger("cal_disp").setLevel(logging.ERROR)
for label, path in cases.items():
    east, north, up = workflow_old._load_los_bands(path)
    before = f"old loader: accepted, arrays {east.shape}"
    try:
        run_calibration(
            disp_file=disp_file,
            unr_grid_latlon_file=lookup_file,
            unr_timeseries_dir=inp / "gnss",
            output_dir=work / "out" / label.replace(" ", "_"),
            los_file=path,
            dem_file=dem_file,
        )
        after = "run_calibration: ran (unexpected)"
    except ValueError as exc:
        after = "run_calibration: ValueError: " + str(exc).replace(str(SCRATCH), "<scratch>")
    print(f"{label}\n    before: {before}\n    after : {after}\n")
    assert after.startswith("run_calibration: ValueError")

geometry check in the old workflow source: False



shifted transform
    before: old loader: accepted, arrays (7733, 9464)
    after : run_calibration: ValueError: LOS file <scratch>/11_input_consistency/shifted_transform/OPERA_L3_DISP-S1-STATIC_F08882_20140403_S1A_v1.0_line_of_sight_enu.tif is not on the grid of DISP file OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20251027T005420Z.nc: transform (origin (72270.0, 3385905.0), pixel (30.0, -30.0)) != (origin (71970.0, 3385920.0), pixel (30.0, -30.0)); origin shifted by (10, 0.5) pixels



different CRS
    before: old loader: accepted, arrays (7733, 9464)
    after : run_calibration: ValueError: LOS file <scratch>/11_input_consistency/other_crs/OPERA_L3_DISP-S1-STATIC_F08882_20140403_S1A_v1.0_line_of_sight_enu.tif is not on the grid of DISP file OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20251027T005420Z.nc: CRS EPSG:32616 != EPSG:32615



other frame id
    before: old loader: accepted, arrays (7733, 9464)
    after : run_calibration: ValueError: LOS file <scratch>/11_input_consistency/other_frame/OPERA_L3_DISP-S1-STATIC_F08622_20140403_S1A_v1.0_line_of_sight_enu.tif is not on the grid of DISP file OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20251027T005420Z.nc: frame id 8622 != 8882



## LOS nodata

The golden LOS raster uses 0.0 as nodata: a third of the grid is outside the swath and holds
all-zero vectors. The old loader returned them as valid numbers; the new one returns NaN.

In [7]:
with rasterio.open(los_file) as src:
    print("LOS nodata value:", src.nodata)

old_bands = workflow_old._load_los_bands(los_file)
new_bands = _load_los_bands(los_file)
zero = (old_bands[0] == 0) & (old_bands[1] == 0) & (old_bands[2] == 0)
print(f"old loader: {int(zero.sum()):,} all-zero LOS vectors of {zero.size:,} pixels ({zero.mean():.1%}), NaN: {int(np.isnan(old_bands[0]).sum())}")
print(f"new loader: {int(((new_bands[0] == 0) & (new_bands[1] == 0) & (new_bands[2] == 0)).sum()):,} all-zero LOS vectors, NaN: {int(np.isnan(new_bands[0]).sum()):,}")
assert np.array_equal(np.isnan(new_bands[0]), zero)
valid = ~zero
assert all(np.array_equal(o[valid], n[valid]) for o, n in zip(old_bands, new_bands))
norm = np.sqrt(sum(b[valid].astype(np.float64) ** 2 for b in new_bands))
print(f"valid vectors are unchanged; their norm is {norm.min():.4f} .. {norm.max():.4f}")

LOS nodata value: 0.0


old loader: 24,789,047 all-zero LOS vectors of 73,185,112 pixels (33.9%), NaN: 0
new loader: 0 all-zero LOS vectors, NaN: 24,789,047


valid vectors are unchanged; their norm is 0.9993 .. 1.0007


In [8]:
# How many of the staged UNR grid points sit on LOS nodata pixels?
from pyproj import Transformer

staged = sorted(int(p.name[:6]) for p in (inp / "gnss").glob("*_IGS20_constant.tenv8"))
lookup = {int(r[0]): (float(r[1]), float(r[2])) for r in (line.split() for line in lookup_file.read_text().splitlines()) if r}
with rasterio.open(los_file) as src:
    to_utm = Transformer.from_crs(4326, src.crs, always_xy=True)
    inv = ~src.transform
inside = on_nodata = 0
for station in staged:
    lon, lat = lookup[station]
    lon = lon - 360 if lon > 180 else lon
    col, row = inv * to_utm.transform(lon, lat)
    r, c = int(row), int(col)
    if 0 <= r < zero.shape[0] and 0 <= c < zero.shape[1]:
        inside += 1
        on_nodata += bool(zero[r, c])
print(f"{len(staged)} staged UNR grid points, {inside} inside the raster, {on_nodata} on LOS nodata pixels")

157 staged UNR grid points, 107 inside the raster, 33 on LOS nodata pixels


Grid points on nodata pixels have no line of sight. Venti already leaves out stations
whose sampled LOS has no horizontal component, so the calibration values of the golden
product do not change; with NaN the absence of data is explicit in cal-disp itself and no
longer depends on that downstream rule.